# FictiPay Customer Churn Prediction Pipeline
**Author**: Shayer Mahmud Sowmik
This notebook implements the complete end-to-end churn prediction pipeline using Polars LazyFrames, LightGBM, and XGBoost.

## 1. Setup & Environment Ingestion

In [ ]:
import os
import datetime
import polars as pl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score
import optuna
import shap

print('All libraries loaded.')

## 2. Polars Ingestion and Out-of-Core Processing (Stage 1 & 2)

In [ ]:
OBSERVATION_END = datetime.date(2024, 3, 31)
KYC_PATH = 'bkash-presents-nsucec-datathon/public/kyc.parquet'
TRX_PATH_PATTERN = 'bkash-presents-nsucec-datathon/public/transactions/trx_2024-0*.parquet'
BALANCE_PATH_PATTERN = 'bkash-presents-nsucec-datathon/public/dayend_balance/balance_2024-0*.parquet'
TRAIN_LABELS_PATH = 'bkash-presents-nsucec-datathon/public/train_labels.csv'
TEST_PATH = 'bkash-presents-nsucec-datathon/public/test.csv'

# Run the features logic using LazyFrames and collect
kyc_lazy = pl.scan_parquet(KYC_PATH).filter(pl.col('ACCOUNT_TYPE') == 'Customer')
kyc_features = kyc_lazy.with_columns([
    ((pl.lit(OBSERVATION_END) - pl.col('ACCOUNT_OPEN_DATE').dt.date()).dt.total_days()).alias('F8_vintage_days'),
    pl.col('GENDER').cast(pl.Categorical).to_physical().cast(pl.Int8),
    pl.col('REGION').cast(pl.Categorical).to_physical().cast(pl.Int16)
])

trx_lazy = pl.scan_parquet(TRX_PATH_PATTERN).with_columns([
    pl.col('TRX_DATETIME').str.slice(0, 10).str.to_date('%Y-%m-%d').alias('TRX_DATE')
])

src_features = (
    trx_lazy.group_by('SRC_ACCOUNT')
    .agg([
        ((pl.lit(OBSERVATION_END) - pl.col('TRX_DATE').max()).dt.total_days()).alias('F1_recency_days'),
        pl.col('TrxID').count().alias('total_tx_count'),
        pl.col('TrxID').filter(pl.col('TRX_DATE').dt.month() == 3).count().alias('tx_count_march'),
        pl.col('TrxID').filter(pl.col('TRX_DATE').dt.month() < 3).count().alias('tx_count_early'),
        pl.col('TRX_AMT').filter(pl.col('TRX_DATE').dt.month() == 3).sum().alias('tx_amt_march'),
        pl.col('TRX_AMT').filter(pl.col('TRX_DATE').dt.month() < 3).sum().alias('tx_amt_early'),
        pl.col('TRX_AMT').filter(pl.col('TRX_TYPE') == 'CashOut').sum().alias('amt_cashout'),
        pl.col('TRX_AMT').filter(pl.col('TRX_TYPE') == 'CashIn').sum().alias('amt_cashin'),
        pl.col('TrxID').filter(pl.col('TRX_TYPE') == 'MerchantPay').count().alias('count_merchant'),
        pl.col('TrxID').filter(pl.col('TRX_TYPE') == 'BillPay').count().alias('count_bill')
    ])
    .rename({'SRC_ACCOUNT': 'ACCOUNT_ID'})
)

inbound_features = (
    trx_lazy.group_by('DST_ACCOUNT')
    .agg([
        pl.col('TrxID').count().alias('F10_inbound_count')
    ])
    .rename({'DST_ACCOUNT': 'ACCOUNT_ID'})
)

balance_lazy = pl.scan_parquet(BALANCE_PATH_PATTERN).with_columns([
    pl.col('DATE').str.to_date('%Y-%m-%d').alias('DATE')
])

balance_features = (
    balance_lazy.group_by('ACCOUNT_ID')
    .agg([
        pl.col('AVAILABLE_BALANCE').filter(pl.col('DATE') == pl.lit(OBSERVATION_END)).first().alias('F9_final_balance'),
        pl.col('AVAILABLE_BALANCE').std().alias('F5_balance_volatility'),
        pl.col('AVAILABLE_BALANCE').filter(pl.col('DATE').dt.month() == 3).mean().alias('avg_bal_march'),
        pl.col('AVAILABLE_BALANCE').filter(pl.col('DATE').dt.month() == 1).mean().alias('avg_bal_jan')
    ])
)

joined_features = (
    kyc_features
    .join(src_features, on='ACCOUNT_ID', how='left')
    .join(inbound_features, on='ACCOUNT_ID', how='left')
    .join(balance_features, on='ACCOUNT_ID', how='left')
)

filled_features = joined_features.with_columns([
    pl.col('F1_recency_days').fill_null(91),
    pl.col('F10_inbound_count').fill_null(0),
    pl.col('total_tx_count').fill_null(0),
    pl.col('tx_count_march').fill_null(0),
    pl.col('tx_count_early').fill_null(0),
    pl.col('tx_amt_march').fill_null(0.0),
    pl.col('tx_amt_early').fill_null(0.0),
    pl.col('amt_cashout').fill_null(0.0),
    pl.col('amt_cashin').fill_null(0.0),
    pl.col('count_merchant').fill_null(0),
    pl.col('count_bill').fill_null(0),
    pl.col('F9_final_balance').fill_null(0.0),
    pl.col('F5_balance_volatility').fill_null(0.0),
    pl.col('avg_bal_march').fill_null(0.0),
    pl.col('avg_bal_jan').fill_null(0.0),
])

final_features = filled_features.with_columns([
    (pl.col('tx_count_march') / ((pl.col('tx_count_early') / 2.0) + 1.0)).alias('F2_momentum_ratio'),
    (pl.col('tx_amt_march') / (pl.col('tx_amt_early') + 1.0)).alias('F3_monetary_trend'),
    (pl.col('avg_bal_march') - pl.col('avg_bal_jan')).alias('F4_balance_depletion'),
    (pl.col('amt_cashout') / (pl.col('amt_cashin') + 1.0)).alias('F6_pass_through_index'),
    ((pl.col('count_merchant') + pl.col('count_bill')) / (pl.col('total_tx_count') + 1.0)).alias('F7_ecosystem_index')
])

train_df = pl.scan_csv(TRAIN_LABELS_PATH).join(final_features, on='ACCOUNT_ID', how='inner').collect(streaming=True).to_pandas()
print('Features extracted successfully. Shape:', train_df.shape)

## 3. Preprocessing: Winsorization & Log Transforms (Stage 3)

In [ ]:
train_df['is_dormant_last_30d'] = (train_df['F1_recency_days'] > 30).astype(int)

numerical_cols = [
    'total_tx_count', 'tx_count_march', 'tx_count_early', 
    'tx_amt_march', 'tx_amt_early', 'amt_cashout', 'amt_cashin',
    'count_merchant', 'count_bill', 'F10_inbound_count', 
    'F9_final_balance', 'F5_balance_volatility', 'avg_bal_march', 'avg_bal_jan',
    'F2_momentum_ratio', 'F3_monetary_trend', 'F6_pass_through_index', 'F7_ecosystem_index'
]

# Winsorization
for col in numerical_cols:
    cap = train_df[col].quantile(0.99)
    train_df[col] = train_df[col].clip(upper=cap)

# Log-Transforms
for col in ['total_tx_count', 'tx_amt_march', 'tx_amt_early', 'F10_inbound_count', 'F9_final_balance', 'F5_balance_volatility']:
    train_df[f'log_{col}'] = np.log1p(train_df[col])

print('Data preprocessed.')

## 4. Modeling & Training Sequence (Stage 4 & 5)

In [ ]:
exclude = ['ACCOUNT_ID', 'ACCOUNT_TYPE', 'ACCOUNT_OPEN_DATE', 'CHURN']
feats = [c for c in train_df.columns if c not in exclude]
X = train_df[feats]
y = train_df['CHURN']

for col in ['GENDER', 'REGION']:
    X[col] = X[col].astype(int)

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(train_df))
imbalance_scale = 519565 / 75435

params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'scale_pos_weight': imbalance_scale,
    'verbose': -1,
    'n_jobs': -1
}

for fold, (t_idx, v_idx) in enumerate(folds.split(X, y)):
    X_tr, y_tr = X.iloc[t_idx], y.iloc[t_idx]
    X_va, y_va = X.iloc[v_idx], y.iloc[v_idx]
    
    train_ds = lgb.Dataset(X_tr, label=y_tr)
    val_ds = lgb.Dataset(X_va, label=y_va)
    
    model = lgb.train(
        params, train_ds, num_boost_round=1000,
        valid_sets=[train_ds, val_ds],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    oof[v_idx] = model.predict(X_va, num_iteration=model.best_iteration)

print('OOF AUC Score:', roc_auc_score(y, oof))